# LifeMate run (Kaggle)

In [ ]:
# Code comes from the private Kaggle Dataset holding the pack_code.sh zip (auto-extracted by Kaggle).
# Attach it plus the competition data dataset; adjust the code dataset path below if named differently.
!cp -r /kaggle/input/lifemate-code/. /kaggle/working/lifemate && ls /kaggle/working/lifemate
%cd /kaggle/working/lifemate

In [ ]:
%pip install -q -r ops/requirements/notebook.txt pytest
# vllm version pinned from uv.lock at authoring time (0.32.0); only on GPU machines.
!nvidia-smi > /dev/null 2>&1 && pip install -q vllm==0.32.0 || echo "no GPU: skipping vllm"
# faster-whisper (ASR mặc định) nằm trong extra gpu cùng vLLM nên không có trong notebook.txt.
!pip install -q faster-whisper==1.2.1
# torchaudio có sẵn trên Kaggle build cho CUDA khác bản torch của vLLM, làm hỏng import transformers.
!pip uninstall -y -q torchaudio


In [ ]:
!python -c "from pathlib import Path; from lifemate.core.envinfo import detect; print(detect(Path('.')))"

In [ ]:
# Danh sách split không có trên git: dựng lại (giống từng byte) nếu thiếu.
!test -f configs/splits/train.txt || python -m lifemate.eval.make_splits --env kaggle --smoke 30


In [ ]:
!python -m pytest -q -m "not data and not tokenizer"

In [ ]:
!python -m lifemate.run.cli --config configs/v0.toml --env kaggle --split train --ids configs/splits/smoke.txt --name smoke

In [ ]:
# Baseline: transcript ASR thật (`--transcripts asr`); v0.toml để gold cho vòng lặp prompt.
# GPU cần từ 24 GB (L4, A30, A100): T4 15 GB không đủ cho vLLM + whisper-large-v3 ở dtype auto.
TAG = "base"
!python -m lifemate.run.cli --config configs/v0.toml --env kaggle --split train --ids configs/splits/val_seen.txt --transcripts asr --name {TAG}_val_seen_asr
!python -m lifemate.run.cli --config configs/v0.toml --env kaggle --split train --ids configs/splits/val_unseen.txt --transcripts asr --name {TAG}_val_unseen_asr


In [ ]:
# Persist runs/ as a new version of a private dataset.
# Create the dataset once: write /kaggle/working/runs/dataset-metadata.json with
#   {"title": "lifemate-runs", "id": "<kaggle-user>/lifemate-runs", "licenses": [{"name": "other"}]}
# then run `kaggle datasets create -p /kaggle/working/runs --dir-mode zip` one time.
!kaggle datasets version -p /kaggle/working/runs -m "runs $(cat VERSION)" --dir-mode zip